In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# סימולציה מדויקת ורועשת עם Qiskit Aer primitives

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח באמצעות הדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או בגרסאות חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
[סימולציה מדויקת עם Qiskit primitives](/guides/simulate-with-qiskit-sdk-primitives) מדגים כיצד להשתמש ב-primitives הייחוסיים הכלולים ב-Qiskit לביצוע סימולציה מדויקת של מעגלים קוונטיים. מעבדי קוונטום קיימים כיום סובלים משגיאות, או רעש, ולכן תוצאות סימולציה מדויקת אינן משקפות בהכרח את התוצאות שהיית מצפה להן בעת הרצת מעגלים על חומרה אמיתית. בעוד ש-primitives הייחוסיים ב-Qiskit אינם תומכים במודל רעש, [Qiskit Aer](https://qiskit.org/ecosystem/aer/) כולל מימושים של ה-primitives שכן תומכים במודל רעש. Qiskit Aer הוא סימולטור מעגלים קוונטיים בעל ביצועים גבוהים שניתן להשתמש בו במקום ה-primitives הייחוסיים לביצועים טובים יותר ותכונות נוספות. הוא חלק מ-[Qiskit Ecosystem](https://qiskit.github.io/ecosystem/). במאמר זה, נדגים את השימוש ב-Qiskit Aer primitives לסימולציה מדויקת ורועשת.

> **Note:** - נדרש `qiskit-aer` גרסה 0.14 ומעלה.
> - בעוד ש-Qiskit Aer primitives מממשים את ממשקי ה-primitive, הם אינם מספקים את אותן האפשרויות כמו Qiskit Runtime primitives. רמת חוסן (Resilience level), לדוגמה, אינה זמינה עם Qiskit Aer primitives.
> - ראה את [תיעוד AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) לפרטים על אפשרויות שיטת הסימולציה ש-Aer תומך בהן.

כדי לחקור סימולציה מדויקת ורועשת, צור מעגל דוגמה על שמונה Qubit:

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

מעגל זה מכיל פרמטרים לייצוג זוויות הסיבוב עבור Gate $R_y$ ו-$R_z$. בעת סימולציה של מעגל זה, עלינו לציין ערכים מפורשים לפרמטרים אלו. בתא הבא, נציין כמה ערכים לפרמטרים אלו ונשתמש ב-Estimator primitive מ-Qiskit Aer לחישוב ערך הציפייה המדויק של האובייקטבל $ZZ \cdots Z$.

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

כעת, נאתחל מודל רעש הכולל שגיאת depolarizing של 2% על כל Gate CX. בפועל, השגיאה הנובעת מ-Gateים דו-קיוביטיים, שהם Gate CX כאן, הם המקור הדומיננטי לשגיאה בעת הרצת מעגל. ראה [בניית מודלי רעש](./build-noise-models) לסקירה של בניית מודלי רעש ב-Qiskit Aer.

בתא הבא, נבנה Estimator המשלב מודל רעש זה ונשתמש בו לחישוב ערך הציפייה של האובייקטבל.

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

כפי שאתה יכול לראות, ערך הציפייה בנוכחות הרעש רחוק מאוד מהערך הנכון. בפועל, תוכל להשתמש במגוון טכניקות של הפחתת שגיאות כדי להתמודד עם השפעות הרעש, אך דיון בטכניקות אלו הוא מעבר לתחום מאמר זה.

כדי לקבל תחושה גסה מאוד של כיצד הרעש משפיע על התוצאה הסופית, שקול את מודל הרעש שלנו, המוסיף שגיאת depolarizing של 2% לכל Gate CX. שגיאת depolarizing עם הסתברות $p$ מוגדרת כערוץ קוונטי $E$ שיש לו את הפעולה הבאה על מטריצת צפיפות $\rho$:

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

כאשר $n$ הוא מספר ה-Qubit, במקרה זה, 2. כלומר, עם הסתברות $p$, המצב מוחלף במצב המעורב לחלוטין, והמצב נשמר עם הסתברות $1 - p$. לאחר $m$ יישומים של ערוץ ה-depolarizing, ההסתברות שהמצב נשמר תהיה $(1 - p)^m$. לכן, אנחנו מצפים שההסתברות לשמירת המצב הנכון בסוף הסימולציה תרד באופן מעריכי עם מספר ה-Gate CX במעגל שלנו.

בואו נספור את מספר ה-Gate CX במעגל שלנו ונחשב $(1 - p)^m$. נקרא ל-`count_ops` כדי לקבל מילון הממפה שמות Gate לספירות, ונאחזר את הערך עבור Gate CX.